# UdaPlay Part 1

Build a persistent vector database from the fifteen supplied game records. Run this notebook before Part 2, from `project/starter`. The database directory is `chromadb` and the collection is `udaplay`.

## Environment

Configure `OPENAI_API_KEY`, `OPENAI_BASE_URL`, and `TAVILY_API_KEY` using an ignored `.env` or exported environment variables. Credentials are never displayed.

In [1]:
import importlib.util
import sys

if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

In [2]:
from pathlib import Path
import json

from lib.udaplay import configure_environment, load_games, GameVectorStore

settings = configure_environment(require_web=False)
print("Required environment variables are configured.")
print("Embedding model:", settings["embedding_model"])

Required environment variables are configured.
Embedding model: text-embedding-3-small


## Load and validate the dataset

Each JSON file becomes a document with the game name, platform, genre, publisher, description and release year. File stems provide stable IDs. Metadata retains the original values and a source path for citations.

In [3]:
records = load_games(Path("games"))
assert len(records) == 15
print("Validated game records:", len(records))
print(json.dumps(records[0], indent=2, ensure_ascii=False))

Validated game records: 15
{
  "id": "001",
  "document": "Name: Gran Turismo\nPlatform: PlayStation 1\nGenre: Racing\nPublisher: Sony Computer Entertainment\nDescription: A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.\nYearOfRelease: 1997",
  "metadata": {
    "Name": "Gran Turismo",
    "Platform": "PlayStation 1",
    "Genre": "Racing",
    "Publisher": "Sony Computer Entertainment",
    "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
    "YearOfRelease": 1997,
    "source": "games/001.json"
  }
}


## Persist the collection

The store configures OpenAI embeddings using the same key, endpoint and model in both notebooks. Upsert makes repeated ingestion safe.

In [4]:
store = GameVectorStore(path="chromadb", create=True)
count = store.ingest(records)
assert count == 15
print(f"Collection {store.collection.name!r}: {count} documents")

Collection 'udaplay': 15 documents


## Reopen and query

Reopen the database to demonstrate persistence. Semantic search returns candidate documents; similarity alone does not establish answer quality. Part 2 evaluates that separately.

In [5]:
reopened = GameVectorStore(path="chromadb")
assert reopened.collection.count() == 15
query = "Who published Grand Theft Auto: San Andreas?"
results = reopened.search(query)
assert results
print("Query:", query)
for result in results:
    print(json.dumps(result.model_dump(), indent=2, ensure_ascii=False))

Query: Who published Grand Theft Auto: San Andreas?
{
  "source_id": "local:002",
  "title": "Grand Theft Auto: San Andreas",
  "content": "Name: Grand Theft Auto: San Andreas\nPlatform: PlayStation 2\nGenre: Action-adventure\nPublisher: Rockstar Games\nDescription: An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson.\nYearOfRelease: 2004",
  "url": "games/002.json",
  "distance": 0.32566869258880615
}
{
  "source_id": "local:003",
  "title": "Gran Turismo 5",
  "content": "Name: Gran Turismo 5\nPlatform: PlayStation 3\nGenre: Racing\nPublisher: Sony Computer Entertainment\nDescription: A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.\nYearOfRelease: 2010",
  "url": "games/003.json",
  "distance": 0.6417862772941589
}
{
  "source_id": "local:001",
  "title": "Gran Turismo",
  "content": "Name: Gran Turismo\nPlatform: PlayStation 1\nGenre: Racing\nPublisher: So

## Part 1 result

The persistent database is ready for Part 2 after all cells above succeed. Keep `chromadb/` locally between runs. It is reproducible from the dataset and is excluded from Git.